## 1. Install Dependencies

In [1]:
!pip install transformers datasets scikit-learn pandas torch


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [2]:
import re
import nltk
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from nltk.corpus import stopwords

## 3. Load Datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load MLMA Arabic hate speech dataset from Hugging Face
print("Loading MLMA dataset...")
mlma_dataset = load_dataset("nedjmaou/MLMA_hate_speech")
mlma = mlma_dataset["train"].to_pandas()
mlma = mlma[["tweet", "sentiment"]].rename(columns={"sentiment": "label"})
mlma = mlma[["tweet", "label"]]

# Load OffensEval 2020 Arabic dataset
print("Loading OffensEval dataset...")
offense_dataset = load_dataset("strombergnlp/offenseval_2020", "ar")
print(f"OffensEval columns: {offense_dataset['train'].column_names}")

offense = offense_dataset["train"].to_pandas()

# Handle OffensEval columns - can vary by config
if "subtask_a" in offense.columns:
    offense = offense[["tweet", "subtask_a"]].rename(columns={"subtask_a": "label"})
elif "label" in offense.columns:
    offense = offense[["text", "label"]].rename(columns={"text": "tweet"})
else:
    # Print available columns for debugging
    print(f"Available columns: {offense.columns.tolist()}")
    print("First row:", offense.iloc[0])
    offense = offense.iloc[:, [0, 1]].copy()
    offense.columns = ["tweet", "label"]

offense = offense[["tweet", "label"]]

# Merge both datasets
df_all = pd.concat([mlma, offense], ignore_index=True)

print(f"MLMA dataset shape: {mlma.shape}")
print(f"OffensEval dataset shape: {offense.shape}")
print(f"Combined dataset shape: {df_all.shape}")
print(f"\nLabel distribution:\n{df_all['label'].value_counts()}")
print(f"\nFirst 5 rows:")
print(df_all.head())

Loading MLMA dataset...
Loading OffensEval dataset...


KeyError: "['label'] not in index"

In [ ]:
# Standardize labels to handle different label values
def standardize_labels(label):
    """Convert different label formats to standard ones"""
    label = str(label).upper().strip()
    if label in ["HATE", "HS"]:
        return "hate"
    elif label in ["OFFENSIVE", "OFF", "HOF"]:
        return "offensive"
    elif label in ["NOT", "NORMAL", "OK"]:
        return "safe"
    else:
        return "safe"  # Default to safe for unknown labels

df_all["label"] = df_all["label"].apply(standardize_labels)
print("\nAfter standardizing labels:")
print(df_all["label"].value_counts())

## 4. Preprocessing

In [ ]:
# Preprocessing function (العربية + English)
stop_words_en = set(stopwords.words("english"))
stop_words_ar = set(stopwords.words("arabic"))

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
    text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation
    text = re.sub(r"\s+", " ", text).strip()  # Remove extra spaces
    # Remove stopwords and short words
    text = " ".join([w for w in text.split() 
                     if w not in stop_words_en and w not in stop_words_ar and len(w) > 1])
    return text

print("\nPreprocessing text...")
df_all["text"] = df_all["tweet"].apply(preprocess)
df_all = df_all[df_all["text"].str.strip() != ""]
df_all = df_all.dropna(subset=["text", "label"])

# Convert labels to integers
label_mapping = {"hate": 0, "offensive": 1, "safe": 2}
df_all["label"] = df_all["label"].map(label_mapping)

print(f"Final dataset shape: {df_all.shape}")
print(f"Label distribution:\n{df_all['label'].value_counts()}")
print(f"\nSample preprocessed texts:")
for i in range(min(3, len(df_all))):
    print(f"  {i+1}. {df_all.iloc[i]['text'][:100]}...")

## 5. Train / Test Split

In [ ]:
# Ensure df_all exists and has the required columns
if 'df_all' not in locals():
    print("Error: df_all not found. Please run cells 6, 7, and 9 first in order.")
else:
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df_all["text"].tolist(),
        df_all["label"].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=df_all["label"]
    )
    
    print(f"Train size: {len(train_texts)}")
    print(f"Test size: {len(test_labels)}")

Error: df_all not found. Please run cells 6, 7, and 9 first in order.


## 6. Tokenization

In [ ]:
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

## 7. Load Model + Weighted Loss



In [ ]:
##Dataset Class
class HateDataset(Dataset):

    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        item = {key: torch.tensor(val[idx]) for key, val in tokenizer(self.texts[idx], truncation=True, padding='max_length', max_length=128).items()}
        item["labels"] = torch.tensor(self.labels[idx])

        return item

In [ ]:
##DataLoader
train_dataset = HateDataset(train_texts, train_labels)
test_dataset = HateDataset(test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3
)

model.to(device)

## 8. Training

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    for batch in train_loader:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()

        optimizer.step()

    print("Epoch finished:", epoch)

## Evaluation

In [ ]:
model.eval()

preds = []
true_labels = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)

        preds.extend(predictions.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["hate", "offensive", "safe"]))        

## 10. Save Model

In [ ]:
model.save_pretrained("hate_speech_ar_model")
tokenizer.save_pretrained("hate_speech_ar_model")

## 11. Audio Pipeline (ASR → Classify)

In [ ]:
import whisper

asr_model = whisper.load_model("base")

result = asr_model.transcribe("audio.wav")

text = result["text"]

print(text)

## 13. Test with gTTS
> **Note:** gTTS saves as MP3. Whisper reads MP3 natively — no conversion needed.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

text = "انا بكره الناس دي"

tts = gTTS(text=text, lang="ar")

tts.save("test_audio.mp3")

Audio("test_audio.mp3")